# Extract features figure

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt

from google.colab import drive
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from scipy.ndimage import gaussian_filter

# ============================================================
# Updated Proposal Figure 1
#
# Fixes:
# 1) larger text
# 2) pooled/mapping panels now reflect unequal dimensionality:
#    SP = 18 pooled features (16 oriented + low + high)
#    TS = 641 pooled features
# ============================================================

drive.mount('/content/drive', force_remount=True)

stim_h5_path = "/content/drive/MyDrive/V2_drift/nsd/stimuli/S1_stimuli_256.h5py"
out_dir = "/content/drive/MyDrive/V1_Drift/plots"
os.makedirs(out_dir, exist_ok=True)

img_id = 28

# ------------------------------------------------------------
# True model dimensionalities for display
# ------------------------------------------------------------
SP_FEATURE_COUNT = 18   # 16 orientation-scale + low-pass + high-pass
TS_FEATURE_COUNT = 641  # before any dimensionality reduction
SP_REGRESSOR_COUNT = 19 # + constant
TS_REGRESSOR_COUNT = 642

# ------------------------------------------------------------
# Font sizes
# ------------------------------------------------------------
FS_COL   = 21
FS_ROW   = 20
FS_SUB   = 17
FS_LAB   = 16
FS_SMALL = 14
FS_TINY  = 13

# ----------------------------
# Helpers
# ----------------------------
def normalize_img(x):
    x = np.asarray(x, dtype=float)
    x -= np.nanmin(x)
    mx = np.nanmax(x)
    if mx > 0:
        x /= mx
    return x


def robust_normalize(x, low=2.0, high=98.0, gamma=1.0):
    x = np.asarray(x, dtype=float)
    lo = np.percentile(x, low)
    hi = np.percentile(x, high)
    if hi <= lo:
        return np.zeros_like(x)
    y = np.clip((x - lo) / (hi - lo), 0, 1)
    if gamma != 1.0:
        y = y ** gamma
    return y


def find_h5_dataset(f):
    for k in ["stimuli", "images", "imgBrick", "imgbrick", "data"]:
        if k in f and isinstance(f[k], h5py.Dataset):
            return k
    for k in f.keys():
        if isinstance(f[k], h5py.Dataset):
            return k
    raise KeyError("Could not find image dataset")


def load_nsd_image_from_h5(h5_path, idx):
    with h5py.File(h5_path, "r") as f:
        ds = f[find_h5_dataset(f)]
        arr = np.asarray(ds[idx])

    if arr.ndim == 3:
        if arr.shape[-1] == 3:
            gray = arr.mean(axis=-1)
        elif arr.shape[0] == 3:
            gray = arr.mean(axis=0)
        else:
            gray = arr.squeeze()
    else:
        gray = arr

    gray = normalize_img(gray)
    rgb = np.dstack([gray, gray, gray])
    return rgb, gray


def make_prf(shape, cx=None, cy=None, sigma=None):
    H, W = shape
    if cx is None:
        cx = int(W * 0.58)
    if cy is None:
        cy = int(H * 0.42)
    if sigma is None:
        sigma = int(H * 0.13)

    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    G = np.exp(-((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * sigma ** 2))
    return G / (G.max() + 1e-12), cx, cy, sigma


def overlay_prf(base_img, prf, alpha=0.65):
    base = robust_normalize(base_img, low=1, high=99, gamma=0.95)
    base_rgb = np.dstack([base, base, base])
    glow = robust_normalize(prf, low=0, high=100, gamma=0.6)[..., None]
    out = base_rgb * (1 - alpha * glow) + 1.0 * (alpha * glow)
    return np.clip(out, 0, 1)


# ----------------------------
# SP-like feature maps
# ----------------------------
def make_gabor_kernel(size=31, theta=0, freq=0.12, sigma=6.0, gamma=0.7):
    ax = np.arange(-(size // 2), size // 2 + 1)
    xx, yy = np.meshgrid(ax, ax)

    th = np.deg2rad(theta)
    x_theta = xx * np.cos(th) + yy * np.sin(th)
    y_theta = -xx * np.sin(th) + yy * np.cos(th)

    env = np.exp(-(x_theta**2 + (gamma**2) * y_theta**2) / (2 * sigma**2))
    carrier = np.cos(2 * np.pi * freq * x_theta)
    ker = env * carrier
    ker -= ker.mean()
    return ker


def conv2_same(img, ker):
    kh, kw = ker.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(img, ((ph, ph), (pw, pw)), mode="reflect")
    out = np.zeros_like(img, dtype=float)

    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            patch = padded[i:i+kh, j:j+kw]
            out[i, j] = np.sum(patch * ker)
    return out


def make_sp_feature_maps(gray_img):
    orientations = [0, 45, 90, 135]
    freqs = [0.06, 0.09, 0.13, 0.18]
    sigmas = [10.0, 8.0, 6.0, 4.5]

    maps = []
    for s in range(4):
        row = []
        for o in range(4):
            ker = make_gabor_kernel(
                size=31,
                theta=orientations[o],
                freq=freqs[s],
                sigma=sigmas[s],
                gamma=0.7
            )
            resp = conv2_same(gray_img, ker)
            resp = np.abs(resp)
            resp = gaussian_filter(resp, sigma=1.0)
            resp = robust_normalize(resp, low=2, high=99, gamma=0.85)
            row.append(resp)
        maps.append(row)
    return maps


# ----------------------------
# TS-like feature maps
# ----------------------------
def local_variance(img, sigma=3):
    mu = gaussian_filter(img, sigma=sigma)
    mu2 = gaussian_filter(img**2, sigma=sigma)
    var = np.maximum(mu2 - mu**2, 0)
    return robust_normalize(var, low=1, high=99, gamma=0.8)


def local_energy(img, sigma=2):
    gx = np.gradient(img, axis=1)
    gy = np.gradient(img, axis=0)
    eng = np.sqrt(gx**2 + gy**2)
    eng = gaussian_filter(eng, sigma=sigma)
    return robust_normalize(eng, low=1, high=99, gamma=0.8)


def local_autocorr_like(img, shift_x=3, shift_y=0, sigma=3):
    shifted = np.roll(img, shift=(shift_y, shift_x), axis=(0, 1))
    prod = img * shifted
    prod = gaussian_filter(prod, sigma=sigma)
    return robust_normalize(prod, low=1, high=99, gamma=0.8)


def make_ts_feature_maps(gray_img):
    img_blur2 = gaussian_filter(gray_img, sigma=2)
    img_blur5 = gaussian_filter(gray_img, sigma=5)

    m11 = local_energy(gray_img, sigma=1)
    m12 = local_energy(gray_img, sigma=3)
    m13 = local_variance(gray_img, sigma=2)
    m14 = local_variance(gray_img, sigma=5)

    m21 = local_autocorr_like(img_blur2, shift_x=2, shift_y=0, sigma=2)
    m22 = local_autocorr_like(img_blur2, shift_x=0, shift_y=2, sigma=2)
    m23 = local_autocorr_like(img_blur2, shift_x=3, shift_y=3, sigma=2)
    m24 = local_autocorr_like(img_blur2, shift_x=4, shift_y=0, sigma=3)

    gx = np.gradient(img_blur2, axis=1)
    gy = np.gradient(img_blur2, axis=0)
    m31 = robust_normalize(gaussian_filter(np.abs(gx), 2), low=1, high=99, gamma=0.8)
    m32 = robust_normalize(gaussian_filter(np.abs(gy), 2), low=1, high=99, gamma=0.8)
    m33 = robust_normalize(gaussian_filter(np.abs(gx * gy), 2), low=1, high=99, gamma=0.8)
    m34 = robust_normalize(gaussian_filter(np.abs(gx) + np.abs(gy), 2), low=1, high=99, gamma=0.8)

    dog1 = gaussian_filter(gray_img, 1) - gaussian_filter(gray_img, 3)
    dog2 = gaussian_filter(gray_img, 2) - gaussian_filter(gray_img, 6)
    m41 = robust_normalize(np.abs(dog1), low=1, high=99, gamma=0.8)
    m42 = robust_normalize(np.abs(dog2), low=1, high=99, gamma=0.8)
    m43 = robust_normalize(np.abs(img_blur2 - img_blur5), low=1, high=99, gamma=0.8)
    m44 = robust_normalize(gaussian_filter(np.abs(dog1 * dog2), 2), low=1, high=99, gamma=0.8)

    return [
        [m11, m12, m13, m14],
        [m21, m22, m23, m24],
        [m31, m32, m33, m34],
        [m41, m42, m43, m44],
    ]


# ----------------------------
# Pooling and responses
# ----------------------------
def pooled_feature_matrix(feature_maps, prf):
    out = np.zeros((4, 4), dtype=float)
    for r in range(4):
        for c in range(4):
            out[r, c] = np.sum(feature_maps[r][c] * prf)
    out = out - out.min()
    if out.max() > 0:
        out = out / out.max()
    return out


def make_response_curve_from_weights(feature_mat, model="SP", n_pts=400):
    if model == "SP":
        amp = 0.92
    else:
        amp = 0.70

    x = np.linspace(0, 1, n_pts)
    peak = amp * np.exp(-((x - 0.17) / 0.10) ** 2)
    undershoot = 0.12 * amp * np.exp(-((x - 0.42) / 0.13) ** 2)
    y = peak - undershoot
    return x, y


# ------------------------------------------------------------
# New display panels for unequal feature counts
# ------------------------------------------------------------
def draw_feature_vector_panel(ax, pooled_mat, n_features, n_show=8, show_ellipsis=True):
    """
    Schematic vector display.
    Uses a few sampled boxes to represent a longer feature vector.
    """
    ax.axis("off")

    vals = pooled_mat.flatten()
    vals = (vals - vals.min()) / (vals.max() - vals.min() + 1e-12)

    idx = np.linspace(0, len(vals) - 1, n_show).astype(int)

    x0 = 0.34
    w = 0.32

    # make box height/gap depend on number shown
    if n_show <= 8:
        h = 0.065
        gap = 0.022
        y0 = 0.18
    elif n_show <= 10:
        h = 0.055
        gap = 0.018
        y0 = 0.18
    else:
        h = 0.048
        gap = 0.014
        y0 = 0.18

    for i, ii in enumerate(idx):
        yy = y0 + (n_show - 1 - i) * (h + gap)
        gray = vals[ii]
        rect = Rectangle(
            (x0, yy), w, h,
            facecolor=str(1 - gray),
            edgecolor="0.65", linewidth=0.9
        )
        ax.add_patch(rect)

    if show_ellipsis:
        ax.text(
            0.5, 0.12, "⋮",
            ha="center", va="center",
            fontsize=34, fontweight="bold", color="0.15"
        )

    ax.text(
        0.5, 0.045, f"{n_features} pooled features",
        ha="center", va="center",
        fontsize=FS_SMALL, fontweight="bold"
    )


def draw_mapping_panel(ax, model_name, n_features, n_regressors, n_show=8, show_ellipsis=True):
    """
    Schematic:
    pooled feature vector × voxel weights -> prediction
    """
    ax.axis("off")

    x_feat = 0.08
    x_w = 0.58
    box_w = 0.18

    # adapt height/gap to number of displayed boxes
    if n_show <= 8:
        box_h = 0.065
        gap = 0.022
        y0 = 0.28
    elif n_show <= 10:
        box_h = 0.055
        gap = 0.018
        y0 = 0.27
    else:
        box_h = 0.048
        gap = 0.014
        y0 = 0.25

    ax.text(0.17, 0.98, "Pooled\nfeatures", ha="center", va="top",
            fontsize=FS_LAB, fontweight="bold")
    ax.text(0.67, 0.98, "Voxel\nweights", ha="center", va="top",
            fontsize=FS_LAB, fontweight="bold")

    for i in range(n_show):
        yy = y0 + (n_show - 1 - i) * (box_h + gap)

        gray_feat = 0.15 + 0.70 * ((n_show - 1 - i) / max(n_show - 1, 1))
        gray_w = 0.15 + 0.70 * (i / max(n_show - 1, 1))

        ax.add_patch(Rectangle(
            (x_feat, yy), box_w, box_h,
            facecolor=str(gray_feat), edgecolor="0.65", linewidth=0.9
        ))

        ax.add_patch(Rectangle(
            (x_w, yy), box_w, box_h,
            facecolor=str(gray_w), edgecolor="0.65", linewidth=0.9
        ))

    ax.text(0.43, 0.54, "×", ha="center", va="center",
            fontsize=22, fontweight="bold")

    if show_ellipsis:
        for y in [0.20, 0.16, 0.12]:
            ax.text(0.17, y, "•", ha="center", va="center",
                    fontsize=18, color="0.2")
            ax.text(0.67, y, "•", ha="center", va="center",
                    fontsize=18, color="0.2")

    # put labels BELOW the panel, not inside it
    ax.text(0.17, -0.02, f"{n_features}",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=FS_SMALL, fontweight="bold",
            clip_on=False)

    ax.text(0.67, -0.02, f"{n_regressors}",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=FS_SMALL, fontweight="bold",
            clip_on=False)

    ax.text(0.43, -0.09, f"{model_name}: features × weights",
            transform=ax.transAxes,
            ha="center", va="center",
            fontsize=FS_TINY, fontstyle="italic", fontweight="bold",
            clip_on=False)


def draw_predicted_response(ax, x, y, color="0.35"):
    ax.plot(x, y, color=color, lw=2.6)

    peak_idx = np.argmax(y)
    x_peak = x[peak_idx]
    y_peak = y[peak_idx]

    ax.annotate(
        "",
        xy=(x_peak, y_peak),
        xytext=(x_peak, 0.0),
        arrowprops=dict(arrowstyle="<->", color=color, lw=1.4)
    )
    ax.set_xlim(0, 1)
    ax.set_ylim(min(-0.18, y.min() - 0.05), max(0.95, y.max() + 0.05))
    ax.axis("off")


# ----------------------------
# Load data
# ----------------------------
rgb_img, gray_img = load_nsd_image_from_h5(stim_h5_path, img_id)

shared_prf, prf_cx, prf_cy, prf_sigma = make_prf(gray_img.shape)

sp_maps = make_sp_feature_maps(gray_img)
ts_maps = make_ts_feature_maps(gray_img)

sp_pooled = pooled_feature_matrix(sp_maps, shared_prf)
ts_pooled = pooled_feature_matrix(ts_maps, shared_prf)

sp_rep_map = sp_maps[1][2]
ts_rep_map = ts_maps[1][1]

sp_rep_overlay = overlay_prf(sp_rep_map, shared_prf, alpha=0.72)
ts_rep_overlay = overlay_prf(ts_rep_map, shared_prf, alpha=0.72)

sp_x, sp_y = make_response_curve_from_weights(sp_pooled, model="SP")
ts_x, ts_y = make_response_curve_from_weights(ts_pooled, model="TS")


# ----------------------------
# Figure
# ----------------------------
fig = plt.figure(figsize=(24, 13), dpi=300)
fig.subplots_adjust(
    left=0.03,
    right=0.985,
    top=0.95,
    bottom=0.05
)

outer = GridSpec(
    3, 6, figure=fig,
    height_ratios=[0.18, 1.0, 1.0],
    width_ratios=[1.15, 2.25, 1.05, 1.00, 1.00, 0.70],
    hspace=0.36,
    wspace=0.42
)

fig.add_artist(FancyArrowPatch(
    (0.055, 0.964), (0.985, 0.964),
    transform=fig.transFigure,
    arrowstyle='-|>', mutation_scale=20, lw=1.5, color='black'
))

titles = [
    "NSD\nimage",
    "Model-derived\nfeature maps",
    "Voxel pRF\nsampling",
    "Pooled\nfeatures",
    "Feature-to-voxel\nmapping",
    "Predicted\nresponse",
]

for i, t in enumerate(titles):
    ax = fig.add_subplot(outer[0, i])
    ax.axis("off")
    ax.text(
        0.5, 0.56, t,
        ha="center", va="center",
        fontsize=FS_COL, fontstyle="italic", fontweight="bold"
    )

# row labels
for ridx, label in zip([1, 2], ["Steerable Pyramid", "Texture Statistics"]):
    ax_lab = fig.add_subplot(outer[ridx, 1])
    ax_lab.text(
        -0.12, 0.5, label,
        transform=ax_lab.transAxes,
        rotation=90,
        ha="center", va="center",
        fontsize=FS_ROW, fontweight="bold"
    )
    ax_lab.axis("off")

# ------------------------------------------------
# Single image spanning both model rows
# ------------------------------------------------
ax_img = fig.add_subplot(outer[1:, 0])
ax_img.imshow(rgb_img)
ax_img.set_xticks([])
ax_img.set_yticks([])
for s in ax_img.spines.values():
    s.set_visible(True)
    s.set_color("0.75")
    s.set_linewidth(1.0)

# ============================================================
# Row 1: SP
# ============================================================
gs_sp_outer = GridSpecFromSubplotSpec(
    1, 2, subplot_spec=outer[1, 1],
    width_ratios=[0.24, 1.0], wspace=0.05
)

ax_sp_lab = fig.add_subplot(gs_sp_outer[0, 0])
ax_sp_lab.axis("off")

scale_labels = ["fine", "med-fine", "med-coarse", "coarse"]
ypos = [0.87, 0.63, 0.39, 0.15]
for y, lab in zip(ypos, scale_labels):
    ax_sp_lab.text(
        0.98, y, lab,
        ha="right", va="center",
        fontsize=FS_SMALL, fontweight="bold"
    )

gs_bank1 = GridSpecFromSubplotSpec(
    4, 4, subplot_spec=gs_sp_outer[0, 1],
    wspace=0.04, hspace=0.04
)

for r in range(4):
    for c in range(4):
        ax = fig.add_subplot(gs_bank1[r, c])
        ax.imshow(sp_maps[r][c], cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)

ax_bb1 = fig.add_subplot(gs_sp_outer[0, 1])
ax_bb1.axis("off")

ax_bb1.text(
    0.50, 1.11, "Example steerable-pyramid feature maps",
    ha="center", va="bottom",
    fontsize=FS_SUB, fontstyle="italic", fontweight="bold"
)

ori_labels = ["0°", "45°", "90°", "135°"]
xpos = [0.12, 0.37, 0.62, 0.87]
for x, lab in zip(xpos, ori_labels):
    ax_bb1.text(
        x, 1.04, lab,
        ha="center", va="bottom",
        fontsize=FS_SMALL, fontweight="bold"
    )

ax_prf1 = fig.add_subplot(outer[1, 2])
ax_prf1.imshow(sp_rep_overlay)
ax_prf1.contour(shared_prf, levels=[0.2, 0.4, 0.6, 0.8], colors="white", linewidths=0.9)
ax_prf1.set_xticks([])
ax_prf1.set_yticks([])
for s in ax_prf1.spines.values():
    s.set_visible(False)

ax_prf1.text(
    0.5, -0.12, "same pRF\napplied to all feature maps",
    transform=ax_prf1.transAxes,
    ha="center", va="top",
    fontsize=FS_SMALL, fontweight="bold"
)

ax_vec1 = fig.add_subplot(outer[1, 3])
draw_feature_vector_panel(
    ax_vec1,
    pooled_mat=sp_pooled,
    n_features=SP_FEATURE_COUNT,
    n_show=6
)

ax_reg1 = fig.add_subplot(outer[1, 4])
draw_mapping_panel(
    ax_reg1,
    model_name="SP",
    n_features=SP_FEATURE_COUNT,
    n_regressors=SP_FEATURE_COUNT,
    n_show=6
)

ax_resp1 = fig.add_subplot(outer[1, 5])
draw_predicted_response(ax_resp1, sp_x, sp_y, color="0.45")

# ============================================================
# Row 2: TS
# ============================================================
gs_ts_outer = GridSpecFromSubplotSpec(
    1, 2, subplot_spec=outer[2, 1],
    width_ratios=[0.24, 1.0], wspace=0.05
)

ax_ts_lab = fig.add_subplot(gs_ts_outer[0, 0])
ax_ts_lab.axis("off")

ts_row_labels = ["local", "spatial", "orientation", "cross-scale"]
for y, lab in zip(ypos, ts_row_labels):
    ax_ts_lab.text(
        0.98, y, lab,
        ha="right", va="center",
        fontsize=FS_SMALL, fontweight="bold"
    )

gs_bank2 = GridSpecFromSubplotSpec(
    4, 4, subplot_spec=gs_ts_outer[0, 1],
    wspace=0.04, hspace=0.04
)

for r in range(4):
    for c in range(4):
        ax = fig.add_subplot(gs_bank2[r, c])
        ax.imshow(ts_maps[r][c], cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
        for s in ax.spines.values():
            s.set_visible(False)

ax_bb2 = fig.add_subplot(gs_ts_outer[0, 1])
ax_bb2.axis("off")

ax_bb2.text(
    0.50, 1.04, "Example texture-statistics feature maps (subset)",
    ha="center", va="bottom",
    fontsize=FS_SUB, fontstyle="italic", fontweight="bold"
)

ax_prf2 = fig.add_subplot(outer[2, 2])
ax_prf2.imshow(ts_rep_overlay)
ax_prf2.contour(shared_prf, levels=[0.2, 0.4, 0.6, 0.8], colors="white", linewidths=0.9)
ax_prf2.set_xticks([])
ax_prf2.set_yticks([])
for s in ax_prf2.spines.values():
    s.set_visible(False)

ax_prf2.text(
    0.5, -0.12, "same pRF\napplied to all feature maps",
    transform=ax_prf2.transAxes,
    ha="center", va="top",
    fontsize=FS_SMALL, fontweight="bold"
)

ax_vec2 = fig.add_subplot(outer[2, 3])
draw_feature_vector_panel(
    ax_vec2,
    pooled_mat=ts_pooled,
    n_features=TS_FEATURE_COUNT,
    n_show=6
)

ax_reg2 = fig.add_subplot(outer[2, 4])
draw_mapping_panel(
    ax_reg2,
    model_name="TS",
    n_features=TS_FEATURE_COUNT,
    n_regressors=TS_FEATURE_COUNT,
    n_show=6
)

ax_resp2 = fig.add_subplot(outer[2, 5])
draw_predicted_response(ax_resp2, ts_x, ts_y, color="0.5")

png_path = os.path.join(out_dir, "proposal_figure1_fixed_counts.png")

plt.savefig(png_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:")
print(png_path)

# Single Voxel Plot

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from google.colab import drive
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

drive.mount('/content/drive')

# ============================================================
# USER SETTINGS
# ============================================================
SAVE_DIR = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
OUT_DIR = "/content/drive/MyDrive/V1_Drift/plots/single_voxels_figures"
os.makedirs(OUT_DIR, exist_ok=True)

TO_ZSCORE=2
CORR_METHOD = "pearson" # options:"pearson"/"spearman"
GOF_METRIC = "cvR2"   # options: "cvR2" / "pearson"

# display / style
FIG_W = 11
FIG_H = 12
HIST_BINS = 20

TITLE_FS = 18
SUBTITLE_FS = 10
LABEL_FS = 10
TICK_FS = 9
CBAR_FS = 8

LINEWIDTH_MEAN = 2.5
LINEWIDTH_SUBJ = 1.2
ALPHA_SUBJ = 0.8

plt.rcParams["font.weight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["axes.titleweight"] = "bold"


# ============================================================
# FILE NAME HELPERS
# ============================================================
def zscore_suffix(to_zscore: int) -> str:
    return {0:"", 1:"_zscore", 2:"_zeroMean", 3:"_equalStd", 4:"_zeroROImean"}.get(to_zscore, "")

OUT_FIG = os.path.join(
    OUT_DIR,
    f"all_rois_{GOF_METRIC}_comparison"
    f"{zscore_suffix(TO_ZSCORE)}"
    f"_{CORR_METHOD}.png"
)

def r2thresh_suffix(r2thresh: float) -> str:
    if r2thresh and r2thresh > 0:
        return f"r2thresh{r2thresh:4.0f}"
    return ""

def build_perm_filename(
    roi: int,
    nperms: int = 1000,
    to_zscore: int = TO_ZSCORE,
    fixedFirst: bool = False,
    toNormalize: bool = False,
    r2thresh: float = 0.0
) -> str:
    fixed_str = "_fixedFirst_" if fixedFirst else ""
    norm_str = "_normalized" if toNormalize else ""
    z_str = zscore_suffix(to_zscore)
    r2_str = r2thresh_suffix(r2thresh)
    rois_str = f"roi{roi}"
    return f"perm_{rois_str}{fixed_str}{nperms}{z_str}{norm_str}{r2_str}.pkl"

def load_perm_for_roi(
    roi: int,
    nperms: int = 1000,
    to_zscore: int = TO_ZSCORE,
    fixedFirst: bool = False,
    r2thresh: float = 0.0,
    save_dir: str = SAVE_DIR
):
    if roi < 1 or roi > 4:
        raise ValueError("roi must be 1..4 (1=V1,2=V2,3=V3,4=hV4)")

    fname = build_perm_filename(
        roi=roi,
        nperms=nperms,
        to_zscore=TO_ZSCORE,
        fixedFirst=fixedFirst,
        r2thresh=r2thresh
    )
    perm_path = os.path.join(save_dir, fname)

    if not os.path.exists(perm_path):
        cand = sorted([
            f for f in os.listdir(save_dir)
            if f.startswith(f"perm_roi{roi}") and f.endswith(".pkl")
        ])[:20]
        raise FileNotFoundError(
            f"Missing: {perm_path}\nCandidates:\n  " + "\n  ".join(cand)
        )

    with open(perm_path, "rb") as f:
        perm = pickle.load(f)

    roi_idx = roi - 1
    subSessions = np.asarray(perm["subSessions"])
    S = int(np.min(np.sum(subSessions, axis=1)))
    subjects = list(perm["subjects"])

    print(f"Loaded ROI {roi} from: {perm_path}")
    print(f"Subjects: {subjects}, minSessions={S}")

    return perm, roi_idx, S, subjects, perm_path


# ============================================================
# ANALYSIS HELPERS
# ============================================================
def safe_corr(
    x,
    y,
    method=CORR_METHOD
):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3:
        return np.nan

    x = x[m]
    y = y[m]

    if method == "pearson":
        return pearsonr(x, y)[0]

    elif method == "spearman":
        return spearmanr(x, y)[0]

    raise ValueError(
        f"Unknown method: {method}"
    )

def compute_roi_gof_stats(perm, roi_idx):
    """
    Reproduce the article-style logic for the cvR² GOF figure:
    - use r2split and r2perm
    - subject-level off-diagonal corr with session distance
    - average across subjects
    - permutation test from permuted session order
    """
    subSessions = np.asarray(perm["subSessions"])
    S = int(np.min(np.sum(subSessions, axis=1)))
    subj_idx = np.arange(len(perm["subjects"]), dtype=int)

    sess = np.arange(1, S + 1)
    sessDiff = sess[:, None] - sess[None, :]
    sessDist = np.abs(sessDiff)
    mask_off = sessDist > 0

    if GOF_METRIC == "cvR2":
        data_split = perm["r2oriSplit"][roi_idx][subj_idx, :S, :S]
        data_perm  = perm["r2oriPerm"][roi_idx][subj_idx, :, :S, :S]
        metric_label = "cvR²"

    elif GOF_METRIC == "pearson":
        data_split = perm["pearsonRori"][roi_idx][subj_idx, :S, :S]
        data_perm  = perm["pearsonOriPerm"][roi_idx][subj_idx, :, :S, :S]
        metric_label = "Pearson r"

    else:
        raise ValueError("GOF_METRIC must be 'cvR2' or 'pearson'")

    nsub = data_split.shape[0]
    nperms = data_perm.shape[1]
    dists_vec = sessDist[mask_off]

    # subjectwise real r, subjectwise perm r, and diagonal-average curve
    sub_r = np.full(nsub, np.nan, float)
    sub_curve = np.full((nsub, S - 1), np.nan, float)
    sub_rperm = np.full((nsub, nperms), np.nan, float)

    for si in range(nsub):
        M = data_split[si]
        sub_r[si] = safe_corr(dists_vec, M[mask_off])

        for d in range(1, S):
            sub_curve[si, d - 1] = np.nanmean(M[sessDist == d])

        PM = data_perm[si]
        for ip in range(nperms):
            Mp = PM[ip]
            sub_rperm[si, ip] = safe_corr(dists_vec, Mp[mask_off])

    r_real = np.nanmean(sub_r)
    r_perm = np.nanmean(sub_rperm, axis=0)
    p_perm = np.mean(r_perm <= r_real)

    # subject-level permutation p-values
    # For drift in cvR², expected direction is negative:
    # stronger drift = more negative correlation with session lag
    sub_p = np.array([
        np.mean(sub_rperm[si, :] <= sub_r[si])
        for si in range(nsub)
    ])

    sub_sig = sub_p < 0.05
    n_sig_subjects = int(np.sum(sub_sig))

    # group matrix: average across subjects, with diagonal blanked for display
    M_group = np.nanmean(data_split, axis=0).copy()
    np.fill_diagonal(M_group, np.nan)

    return {
        "S": S,
        "nsub": nsub,
        "nperms": nperms,
        "matrix": M_group,
        "curve_sub": sub_curve,
        "curve_mean": np.nanmean(sub_curve, axis=0),
        "r_real": r_real,
        "r_perm": r_perm,
        "p_perm": p_perm,
        # subject-level stats
        "sub_r": sub_r,
        "sub_rperm": sub_rperm,
        "sub_p": sub_p,
        "sub_sig": sub_sig,
        "n_sig_subjects": n_sig_subjects,
        "metric_label": metric_label,
        }

def format_perm_p_for_title(p_perm, nperms, decimals=3):
    """
    Format permutation p-value for figure titles so it never shows p=0.
    For nperms=1000:
      - if p_perm == 0, show 'p<0.001'
      - otherwise show e.g. 'p=0.013'
    """
    min_p = 1.0 / nperms

    if p_perm <= 0:
        return f"p<{min_p:.{decimals}f}"
    else:
        return f"p={p_perm:.{decimals}f}"

# ============================================================
# PLOTTING
# ============================================================
roi_labels = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}

all_stats = {}
all_mats = []

# First pass: load everything
for roi in [1, 2, 3, 4]:
    perm, roi_idx, S, subjects, perm_path = load_perm_for_roi(
        roi=roi,
        nperms=1000,
        to_zscore=TO_ZSCORE,
        fixedFirst=False,
        r2thresh=0.0,
        save_dir=SAVE_DIR
    )
    stats = compute_roi_gof_stats(perm, roi_idx)
    all_stats[roi] = stats
    all_mats.append(stats["matrix"])

# shared matrix color scale across ROIs
global_vmin = np.nanmin([np.nanmin(m) for m in all_mats])
global_vmax = np.nanmax([np.nanmax(m) for m in all_mats])

# histogram x-range shared across ROIs
all_perm_vals = np.concatenate([all_stats[r]["r_perm"] for r in [1, 2, 3, 4]])
all_real_vals = np.array([all_stats[r]["r_real"] for r in [1, 2, 3, 4]])
hist_xmin = min(np.nanmin(all_perm_vals), np.nanmin(all_real_vals)) - 0.03
hist_xmax = max(np.nanmax(all_perm_vals), np.nanmax(all_real_vals)) + 0.03

# curve y-range shared across ROIs
all_curve_vals = np.concatenate([all_stats[r]["curve_sub"].ravel() for r in [1, 2, 3, 4]])
curve_ymin = np.nanmin(all_curve_vals) - 0.01
curve_ymax = np.nanmax(all_curve_vals) + 0.01

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    constrained_layout=False
    )

gs = fig.add_gridspec(
    nrows=4,
    ncols=3,
    width_ratios=[1, 1, 1],   # matrix | curve | hist
    wspace=0.01,
    hspace=0.55
    )

for row, roi in enumerate([1, 2, 3, 4]):
    ax_mat = fig.add_subplot(gs[row, 0])
    ax_curve = fig.add_subplot(gs[row, 1])
    ax_hist = fig.add_subplot(gs[row, 2])

    for ax in [ax_mat, ax_curve, ax_hist]:
        ax.set_box_aspect(1)

    stats = all_stats[roi]
    M = stats["matrix"]
    S = stats["S"]
    x = np.arange(1, S)

    # -------------------------
    # LEFT: GOF matrix
    # -------------------------
    im = ax_mat.imshow(
        M,
        origin="upper",
        aspect="auto",
        vmin=global_vmin,
        vmax=global_vmax
    )

    ax_mat.set_title(
    f"{roi_labels[roi]} | {stats['metric_label']} matrix",
    fontsize=SUBTITLE_FS,
    fontweight="bold"
    )
    ax_mat.set_xlabel("test session", fontsize=LABEL_FS, fontweight="bold")
    ax_mat.set_ylabel("train session", fontsize=LABEL_FS, fontweight="bold")
    ax_mat.set_xticks([0, S - 1])
    ax_mat.set_xticklabels([1, S], fontsize=TICK_FS, fontweight="bold")
    ax_mat.set_yticks([0, S - 1])
    ax_mat.set_yticklabels([1, S], fontsize=TICK_FS, fontweight="bold")

    ax_cbar = inset_axes(
        ax_mat,
        width="5%",
        height="100%",
        loc="lower left",
        bbox_to_anchor=(1.05, 0, 1, 1),
        bbox_transform=ax_mat.transAxes,
        borderpad=0
    )

    cbar = fig.colorbar(im, cax=ax_cbar)
    # move colorbar closer to heatmap

    cbar.ax.tick_params(labelsize=CBAR_FS, length=2)
    for t in cbar.ax.get_yticklabels():
        t.set_fontweight("bold")

    # -------------------------
    # MIDDLE: cvR² vs session lag
    # -------------------------
    for si in range(stats["curve_sub"].shape[0]):
        ax_curve.plot(
            x,
            stats["curve_sub"][si],
            linewidth=LINEWIDTH_SUBJ,
            alpha=ALPHA_SUBJ
        )

    ax_curve.plot(
        x,
        stats["curve_mean"],
        color="k",
        linewidth=LINEWIDTH_MEAN
    )

    p_text = format_perm_p_for_title(
        stats["p_perm"],
        stats["nperms"],
        decimals=3
    )

    if stats["p_perm"] < 0.001:
        sig_txt = "***"
    elif stats["p_perm"] < 0.01:
        sig_txt = "**"
    elif stats["p_perm"] < 0.05:
        sig_txt = "*"
    else:
        sig_txt = "n.s."

    ax_curve.set_title(
        f"{roi_labels[roi]} | {stats['metric_label']}: r={stats['r_real']:.2f}  {sig_txt}\n"
        f"{p_text} | sig subjects: {stats['n_sig_subjects']}/{stats['nsub']}",
        fontsize=SUBTITLE_FS,
        fontweight="bold"
    )
    ax_curve.set_xlabel(r"$\Delta$ session", fontsize=LABEL_FS, fontweight="bold")
    ax_curve.set_ylabel(stats["metric_label"], fontsize=LABEL_FS, fontweight="bold")
    ax_curve.set_xlim(1, S - 1)
    ax_curve.set_ylim(curve_ymin, curve_ymax)
    ax_curve.set_xticks([1, S - 1])
    ax_curve.set_xticklabels([1, S - 1], fontsize=TICK_FS, fontweight="bold")
    ax_curve.tick_params(axis="y", labelsize=TICK_FS)
    for t in ax_curve.get_yticklabels():
        t.set_fontweight("bold")

    # -------------------------
    # RIGHT: permutation histogram
    # -------------------------
    counts, edges = np.histogram(stats["r_perm"], bins=HIST_BINS)
    prob = counts / counts.sum()

    ax_hist.bar(
        edges[:-1],
        prob,
        width=np.diff(edges),
        align="edge",
        color=[0.75, 0.75, 0.75],
        edgecolor="none"
    )
    ax_hist.axvline(stats["r_real"], color="k", linewidth=LINEWIDTH_MEAN)

    ax_hist.set_title(
    f"{roi_labels[roi]} | permutations",
    fontsize=SUBTITLE_FS,
    fontweight="bold"
    )
    ax_hist.set_xlabel(f"{CORR_METHOD.capitalize()} correlation (r)", fontsize=LABEL_FS, fontweight="bold")
    ax_hist.set_ylabel("permutations prob.", fontsize=LABEL_FS, fontweight="bold")
    ax_hist.set_xlim(hist_xmin, hist_xmax)
    ax_hist.tick_params(axis="both", labelsize=TICK_FS)
    for t in ax_hist.get_xticklabels() + ax_hist.get_yticklabels():
        t.set_fontweight("bold")


NORMALIZATION_LABELS = {
    0: "",
    1: "zscore",
    2: "zeroMean",
    3: "equalStd",
    4: "zeroROImean"
}

fig.suptitle(
    f"Cross-session goodness-of-fit ({GOF_METRIC}) across visual areas",
    y=0.995,
    fontsize=18,
    fontweight="bold"
)

if TO_ZSCORE != 0:
    fig.text(
        0.02,
        0.97,
        f"Normalization: {NORMALIZATION_LABELS[TO_ZSCORE]}",
        ha="left",
        va="top",
        fontsize=12,
        fontweight="bold"
    )

fig.subplots_adjust(
    left=0.06,
    right=0.96,
    bottom=0.04,
    top=0.93,
    wspace=0.01
)

plt.savefig(OUT_FIG, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure to:", OUT_FIG)

# comparing drift in the rois

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1) Add slope computation, reusing YOUR sessDist + mask_off
# ============================================================

def safe_slope_ols(x, y, add_intercept=True):
    """
    OLS slope for y ~ x (+ intercept), ignoring NaNs/Infs.
    Returns slope (beta1).
    """
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan

    xm = x[m].astype(float)
    ym = y[m].astype(float)

    if add_intercept:
        X = np.column_stack([np.ones_like(xm), xm])
        b, *_ = np.linalg.lstsq(X, ym, rcond=None)
        return float(b[1])
    else:
        denom = np.sum(xm**2)
        if denom == 0:
            return np.nan
        return float(np.sum(xm * ym) / denom)


def subjectwise_distcorr_and_slope(sim_data, sim_perm=None):
    """
    Per-subject:
      r_sub     = corr(distVec, simVec_offdiag)
      slope_sub = slope in simVec_offdiag ~ distVec (OLS with intercept)
    Optionally returns p_sub if sim_perm is provided (same one-sided logic as you used).
    """
    nsub = sim_data.shape[0]
    dists_vec = sessDist[mask_off].astype(float)

    r_sub = np.full(nsub, np.nan, float)
    slope_sub = np.full(nsub, np.nan, float)

    # optional perm p-values (per-subject)
    p_sub = None
    if sim_perm is not None:
        nperms_local = sim_perm.shape[1]
        p_sub = np.full(nsub, np.nan, float)

    for si in range(nsub):
        M = sim_data[si]
        y_vec = M[mask_off].astype(float)

        # Option 1: correlation (already in your code)
        r_sub[si] = safe_corr(dists_vec, y_vec)

        # Option 2: slope of regression line (cvR² ~ lag)
        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        # Optional permutation p-value for r (same as your subjectwise_distcorr)
        if sim_perm is not None:
            r_perm_sub = np.full(nperms_local, np.nan, float)
            for ip in range(nperms_local):
                Mp = sim_perm[si, ip]
                r_perm_sub[ip] = safe_corr(dists_vec, Mp[mask_off])
            p_sub[si] = float(np.mean(r_perm_sub <= r_sub[si]))  # one-sided (more negative)

    return r_sub, slope_sub, p_sub


def slope_perm_subjectwise(sim_data, sim_perm):
    """
    Permutation test for the GROUP-LEVEL mean slope across subjects.
    Uses the same permuted matrices you already loaded (sim_perm).

    Returns:
      slope_sub     : (nsub,) real slope per subject
      slope_real    : scalar, mean(slope_sub)
      slope_perm    : (nperms,) mean slope across subjects for each permutation
      p_perm        : scalar, one-sided p-value for slope being MORE negative than null
                      (i.e., drift = negative slope)
    """
    nsub = sim_data.shape[0]
    nperms = sim_perm.shape[1]

    dists_vec = sessDist[mask_off].astype(float)

    slope_sub = np.full(nsub, np.nan, float)
    slope_perm_sub = np.full((nsub, nperms), np.nan, float)

    for si in range(nsub):
        M = sim_data[si]
        y_vec = M[mask_off].astype(float)

        # real slope
        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        # perm slopes
        for ip in range(nperms):
            Mp = sim_perm[si, ip]
            y_perm = Mp[mask_off].astype(float)
            slope_perm_sub[si, ip] = safe_slope_ols(dists_vec, y_perm, add_intercept=True)

    slope_real = float(np.nanmean(slope_sub))
    slope_perm = np.nanmean(slope_perm_sub, axis=0)  # mean across subjects per permutation

    # one-sided: drift = more negative slope than null
    p_perm = float(np.mean(slope_perm <= slope_real))

    return slope_sub, slope_real, slope_perm, p_perm


# ============================================================
# 2) Helper: bar + dots plot for V1–V4
# ============================================================
def bar_with_subject_dots(ax, values_by_roi, roi_names, ylabel, title):

    nroi = len(values_by_roi)
    nsub = len(values_by_roi[0])

    means = [np.nanmean(v) for v in values_by_roi]
    sems = [
        np.nanstd(v, ddof=1) / np.sqrt(np.sum(np.isfinite(v)))
        for v in values_by_roi
    ]

    x = np.arange(nroi)

    # ------------------------
    # Mean ± SEM
    # ------------------------
    ax.bar(
        x,
        means,
        yerr=sems,
        capsize=4,
        facecolor="none",
        edgecolor="black",
        linewidth=2,
        zorder=1
    )

    # ------------------------
    # Subject colors
    # ------------------------
    cmap = plt.get_cmap("tab10")
    subject_colors = [cmap(i) for i in range(nsub)]

    data_matrix = np.column_stack(values_by_roi)

    rng = np.random.default_rng(0)

    jitter = rng.uniform(-0.08, 0.08, size=(nsub, nroi))

    # ------------------------
    # NEW: connect subjects
    # ------------------------
    for si in range(nsub):

        x_sub = x + jitter[si]
        y_sub = data_matrix[si]

        valid = np.isfinite(y_sub)

        ax.plot(
            x_sub[valid],
            y_sub[valid],
            color=subject_colors[si],
            linewidth=2,
            alpha=0.55,
            zorder=2
        )

    # ------------------------
    # Plot dots
    # ------------------------
    for roi_idx in range(nroi):

        for si in range(nsub):

            ax.scatter(
                x[roi_idx] + jitter[si, roi_idx],
                data_matrix[si, roi_idx],
                s=70,
                color=subject_colors[si],
                edgecolor="black",
                linewidth=0.8,
                zorder=3
            )

    ax.set_xticks(x)

    ax.set_xticklabels(
        roi_names,
        fontsize=14,
        fontweight="bold"
    )

    ax.set_ylabel(
        ylabel,
        fontsize=16,
        fontweight="bold"
    )

    ax.set_title(
        title,
        fontsize=18,
        fontweight="bold"
    )

    ax.tick_params(
        axis="both",
        labelsize=13
    )

    ax.axhline(
        0,
        color="black",
        linewidth=1.5
    )




# ============================================================
# 3) Example: compute BOTH metrics for ALL ROIs and plot
# ============================================================

# If your pipeline already has a function load_perm_for_roi(roi=..., ...)
# we can loop roi=1..4 and reuse exactly your stored arrays.

roi_labels = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}
roi_list = [1, 2, 3, 4]

# Choose which similarity matrix you want: r2split (cvR²) or pearsonRori
use_cvR2 = True   # set False to use Pearson similarity drift instead

r_metric_by_roi = []       # Option 1: r(dist, cvR²)
slope_metric_by_roi = []   # Option 2: slope(cvR² ~ dist)
p_metric_by_roi = []       # optional per-subject p-values (if you pass perms)

for roi in roi_list:
    perm, roi_idx, minSessions, sub_idx, perm_path = load_perm_for_roi(
        roi=roi,
        nperms=1000,
        to_zscore=0,
        fixedFirst=False,
        r2thresh=0.0,
    )

    # As in your code: select consistent S=minSessions across included subjects
    subjects = perm["subjects"]
    subj_list = perm["subjects"]
    subj_to_idx = {s: i for i, s in enumerate(subj_list)}
    sub_idx = np.array([subj_to_idx[s] for s in subjects], dtype=int)

    subSessions = perm["subSessions"][sub_idx, :]
    S = int(np.min(np.sum(subSessions, axis=1)))

    # rebuild sessDist/mask_off for THIS S (to avoid mismatch across ROIs)
    sess = np.arange(1, S + 1)
    sessDist = np.abs(sess[:, None] - sess[None, :])
    mask_off = sessDist > 0

    # pull matrices
    if use_cvR2:
        sim_data = perm["r2oriSplit"][roi_idx][sub_idx, :S, :S]               # (nsub,S,S)
        sim_perm = perm["r2oriPerm"][roi_idx][sub_idx, :, :S, :S]             # (nsub,nperms,S,S)
        metric_name = "cvR²"
    else:
        sim_data = perm["pearsonROri"][roi_idx][sub_idx, :S, :S]
        sim_perm = perm["pearsonOriPerm"][roi_idx][sub_idx, :, :S, :S]
        metric_name = "Pearson r"

    r_sub, slope_sub, p_sub = subjectwise_distcorr_and_slope(sim_data, sim_perm)
    # permutation p-value for SLOPE (group-level, like your correlation perm test)
    slope_sub2, slope_real_mean, slope_perm_mean, slope_p_perm = slope_perm_subjectwise(sim_data, sim_perm)

    print(f"{roi_labels[roi]} | mean slope={slope_real_mean:.6g} | slope perm p(one-sided)={slope_p_perm:.4g}")


    r_metric_by_roi.append(r_sub)
    slope_metric_by_roi.append(slope_sub)
    p_metric_by_roi.append(p_sub)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bar_with_subject_dots(
    axes[0],
    r_metric_by_roi,
    [roi_labels[r] for r in roi_list],
    ylabel=f"corr({metric_name}, session lag)",
    title=f"Correlation({metric_name}, lag)"
)

bar_with_subject_dots(
    axes[1],
    slope_metric_by_roi,
    [roi_labels[r] for r in roi_list],
    ylabel=f"slope of {metric_name} ~ session lag",
    title=f"Slope({metric_name} ~ lag)"
)
from matplotlib.lines import Line2D

nsub = len(r_metric_by_roi[0])
cmap = plt.get_cmap("tab10")

legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor=cmap(i),
        markeredgecolor='black',
        markersize=9,
        label=f"Subject {i+1}"
    )
    for i in range(nsub)
]

fig.legend(
    handles=legend_handles,
    title="Subjects",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.06),
    ncol=4,
    frameon=False,
    fontsize=14,
    title_fontsize=16
)

# Make legend title bold
fig.legends[0].get_title().set_fontweight("bold")

# Force subject labels to NOT be bold
for txt in fig.legends[0].get_texts():
    txt.set_fontweight("normal")

plt.tight_layout(rect=[0,0,1,0.90])
import os

os.makedirs(OUT_DIR, exist_ok=True)

save_path = os.path.join(OUT_DIR, "ROI_drift_comparison_subjectcolors_meanline.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print("Saved figure to:", save_path)

plt.show()



# Population Plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from google.colab import drive
from mpl_toolkits.axes_grid1.inset_locator import inset_axes


drive.mount("/content/drive", force_remount=True)
# =========================================================
# CONFIG
# =========================================================
BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
FIG_DIR = "/content/drive/MyDrive/V1_Drift/plots/population_figures"
os.makedirs(FIG_DIR, exist_ok=True)

TO_ZSCORE = 3
CORR_METHOD = "pearson" # pearson or spearman
NPERMS_WANT = 1000
VREGS = [1, 2, 3, 4]
VREG_NAMES = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}
SUBJECTS = list(range(1, 9))

FIG_W = 11
FIG_H = 12
HIST_BINS = 20

LINE_WIDE = 2.5
LINE_NARROW = 1.2

TITLE_FS = 18
SUBTITLE_FS = 10
LABEL_FS = 10
TICK_FS = 9
CBAR_FS = 8

RDM_COMBOS = [
    ("correlation", "spearman")
]
# =========================================================
# HELPERS
# =========================================================
def _zstr(z):
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][z]

def metric_tag(rdm_build_metric, rdm_compare_metric):
    build_tag = {
        "correlation": "rdmCorrDist",
        "euclidean": "rdmEuclidean"
    }[rdm_build_metric]

    compare_tag = {
        "spearman": "cmpSpearman",
        "rmse": "cmpRMSE"
    }[rdm_compare_metric]

    return f"{build_tag}_{compare_tag}"


def perm_pop_path(base_folder, vreg, isub, to_zscore, nperms,
                  rdm_build_metric, rdm_compare_metric):

    tag = metric_tag(rdm_build_metric, rdm_compare_metric)

    return os.path.join(
        base_folder,
        f"permPop_N{nperms}_v{vreg}_sub{isub}"
        f"{_zstr(to_zscore)}_{tag}.npz"
    )

def safe_corr(x, y, method=CORR_METHOD):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3:
        return np.nan

    x = x[m]
    y = y[m]

    if method == "pearson":
        return pearsonr(x, y)[0]

    elif method == "spearman":
        return spearmanr(x, y)[0]

    raise ValueError(f"Unknown method: {method}")

def matrix_distance_correlation(mat, dist_matrix):
    """
    Correlate off-diagonal matrix entries with session distance.
    """
    mask = np.isfinite(mat) & (dist_matrix > 0)
    if mask.sum() < 3:
        return np.nan
    return safe_corr(dist_matrix[mask], mat[mask])

def load_region_group(vreg,
                      rdm_build_metric,
                      rdm_compare_metric,
                      subjects=SUBJECTS,
                      base_folder=BASE_FOLDER,
                      to_zscore=TO_ZSCORE,
                      nperms=NPERMS_WANT):
    """
    Load all subjects for one visual region from flat permPop files.
    """
    mats = []
    lag_curves = []
    perm_mats = []
    found_subjects = []
    min_sessions = None

    for isub in subjects:
        path = perm_pop_path(base_folder, vreg, isub, to_zscore, nperms,
                             rdm_build_metric,
                             rdm_compare_metric
                             )
        if not os.path.exists(path):
            print(f"[WARN] Missing: {path}")
            continue

        d = np.load(path, allow_pickle=True)

        mat = np.array(d["avgImgCorrMatOri"], dtype=float)
        lag = np.array(d["betweenSessImgOri"], dtype=float)
        pmat = np.array(d["imgCorrMatOriPerm"], dtype=float)
        ns = int(d["minSessions"])

        if min_sessions is None:
            min_sessions = ns
        else:
            min_sessions = min(min_sessions, ns)

        mats.append(mat)
        lag_curves.append(lag)
        perm_mats.append(pmat)
        found_subjects.append(isub)

    if len(found_subjects) == 0:
        raise RuntimeError(f"No files found for vreg={vreg}")

    # crop everything to common session count if needed
    mats = [m[:min_sessions, :min_sessions] for m in mats]
    lag_curves = [l[:min_sessions - 1] for l in lag_curves]
    perm_mats = [p[:, :min_sessions, :min_sessions] for p in perm_mats]

    mats = np.stack(mats, axis=0)              # (nsub, ns, ns)
    lag_curves = np.stack(lag_curves, axis=0)  # (nsub, ns-1)
    perm_mats = np.stack(perm_mats, axis=0)    # (nsub, nperms, ns, ns)

    dist_matrix = np.abs(np.subtract.outer(np.arange(min_sessions), np.arange(min_sessions)))

    return {
        "subjects": np.array(found_subjects),
        "ns": min_sessions,
        "dist_matrix": dist_matrix,
        "avgImgCorrMat_S": mats,
        "betweenSessImg_S": lag_curves,
        "imgCorrMatPerm_S": perm_mats,
    }


# =========================================================
# PLOT FIGURE FOR ALL RDM COMBOS
# =========================================================

for rdm_build_metric, rdm_compare_metric in RDM_COMBOS:

    tag = metric_tag(rdm_build_metric, rdm_compare_metric)

    print("\n" + "=" * 80)
    print(f"Plotting population figure for: {tag}")
    print("=" * 80)

    # -----------------------------------------------------
    # First pass: load all regions and compute global scales
    # -----------------------------------------------------
    all_aggs = {}
    all_mats = []
    all_lag_vals = []
    all_perm_vals = []
    all_obs_vals = []

    for vreg in VREGS:
        agg = load_region_group(
            vreg=vreg,
            rdm_build_metric=rdm_build_metric,
            rdm_compare_metric=rdm_compare_metric
        )
        all_aggs[vreg] = agg

        mats = agg["avgImgCorrMat_S"]
        mean_mat = np.nanmean(mats, axis=0)
        mat_for_show = mean_mat.copy()
        np.fill_diagonal(mat_for_show, np.nan)
        all_mats.append(mat_for_show)

        all_lag_vals.append(agg["betweenSessImg_S"].ravel())

        dist_matrix = agg["dist_matrix"]
        perm_mats = agg["imgCorrMatPerm_S"]
        nperms = perm_mats.shape[1]
        subjects = agg["subjects"]

        subj_r = np.array([
            matrix_distance_correlation(m, dist_matrix)
            for m in mats
        ])
        obs_r = np.nanmean(subj_r)
        all_obs_vals.append(obs_r)

        perm_r = np.full((nperms,), np.nan)
        for ip in range(nperms):
            subj_perm_r = np.array([
                matrix_distance_correlation(perm_mats[si, ip], dist_matrix)
                for si in range(len(subjects))
            ])
            perm_r[ip] = np.nanmean(subj_perm_r)

        all_perm_vals.append(perm_r)

    global_vmin = np.nanmin([np.nanmin(m) for m in all_mats])
    global_vmax = np.nanmax([np.nanmax(m) for m in all_mats])

    all_lag_vals = np.concatenate(all_lag_vals)
    curve_ymin = np.nanmin(all_lag_vals) - 0.03
    curve_ymax = np.nanmax(all_lag_vals) + 0.03

    all_perm_flat = np.concatenate(all_perm_vals)
    all_obs_vals = np.array(all_obs_vals)
    hist_xmin = min(np.nanmin(all_perm_flat), np.nanmin(all_obs_vals)) - 0.03
    hist_xmax = max(np.nanmax(all_perm_flat), np.nanmax(all_obs_vals)) + 0.03

    # -----------------------------------------------------
    # Figure layout: square panels, equal size
    # -----------------------------------------------------
    fig = plt.figure(
        figsize=(FIG_W, FIG_H),
        constrained_layout=False
    )

    gs = fig.add_gridspec(
        nrows=len(VREGS),
        ncols=3,
        width_ratios=[1, 1, 1],
        wspace=0.05,
        hspace=0.55
    )

    for row, vreg in enumerate(VREGS):

        agg = all_aggs[vreg]

        subjects = agg["subjects"]
        ns = agg["ns"]
        dist_matrix = agg["dist_matrix"]
        mats = agg["avgImgCorrMat_S"]
        lag_curves = agg["betweenSessImg_S"]
        perm_mats = agg["imgCorrMatPerm_S"]
        nperms = perm_mats.shape[1]

        axB = fig.add_subplot(gs[row, 0])
        axC = fig.add_subplot(gs[row, 1])
        axD = fig.add_subplot(gs[row, 2])

        for ax in [axB, axC, axD]:
            ax.set_box_aspect(1)

        # -------------------------
        # LEFT: population matrix
        # -------------------------
        mean_mat = np.nanmean(mats, axis=0)
        mat_for_show = mean_mat.copy()
        np.fill_diagonal(mat_for_show, np.nan)

        im = axB.imshow(
            mat_for_show,
            origin="upper",
            aspect="equal",
            cmap="viridis",
            vmin=global_vmin,
            vmax=global_vmax
        )

        axB.set_title(
            f"{VREG_NAMES[vreg]} | population matrix",
            fontsize=SUBTITLE_FS,
            fontweight="bold"
        )
        axB.set_xlabel("session", fontsize=LABEL_FS, fontweight="bold")
        axB.set_ylabel("session", fontsize=LABEL_FS, fontweight="bold")
        axB.set_xticks([0, ns - 1])
        axB.set_xticklabels(["1", str(ns)], fontsize=TICK_FS, fontweight="bold")
        axB.set_yticks([0, ns - 1])
        axB.set_yticklabels(["1", str(ns)], fontsize=TICK_FS, fontweight="bold")

        ax_cbar = inset_axes(
            axB,
            width="5%",
            height="100%",
            loc="lower left",
            bbox_to_anchor=(1.05, 0, 1, 1),
            bbox_transform=axB.transAxes,
            borderpad=0
        )

        cbar = fig.colorbar(im, cax=ax_cbar)
        cbar.ax.tick_params(labelsize=CBAR_FS, length=2)
        for t in cbar.ax.get_yticklabels():
            t.set_fontweight("bold")

        # -------------------------
        # MIDDLE: lag curve
        # -------------------------
        lags = np.arange(1, ns)

        for i in range(lag_curves.shape[0]):
            axC.plot(
                lags,
                lag_curves[i],
                linewidth=LINE_NARROW,
                alpha=0.8
            )

        mean_curve = np.nanmean(lag_curves, axis=0)
        axC.plot(
            lags,
            mean_curve,
            color="black",
            linewidth=LINE_WIDE
        )

        subj_r = np.array([
            matrix_distance_correlation(m, dist_matrix)
            for m in mats
        ])
        obs_r = np.nanmean(subj_r)

        perm_r = np.full((nperms,), np.nan)
        subj_perm_r_all = np.full((len(subjects), nperms), np.nan)

        for si in range(len(subjects)):
            for ip in range(nperms):
                subj_perm_r_all[si, ip] = matrix_distance_correlation(
                    perm_mats[si, ip],
                    dist_matrix
                )

        for ip in range(nperms):
            perm_r[ip] = np.nanmean(subj_perm_r_all[:, ip])

        sub_p = np.array([
            np.mean(subj_perm_r_all[si, :] <= subj_r[si])
            for si in range(len(subjects))
        ])
        n_sig_subjects = int(np.sum(sub_p < 0.05))

        k = np.sum(perm_r <= obs_r)
        p_perm = (k + 1) / (nperms + 1)

        if k == 0 or p_perm < 0.001:
            p_txt = "p < 0.001"
        else:
            p_txt = f"p = {p_perm:.3f}"

        if p_perm < 0.001:
            sig_txt = "***"
        elif p_perm < 0.01:
            sig_txt = "**"
        elif p_perm < 0.05:
            sig_txt = "*"
        else:
            sig_txt = "n.s."

        axC.set_title(
            f"{VREG_NAMES[vreg]} | r={obs_r:.2f} {sig_txt}\n"
            f"{p_txt} | sig subjects: {n_sig_subjects}/{len(subjects)}",
            fontsize=SUBTITLE_FS,
            fontweight="bold"
        )
        axC.set_xlabel(r"$\Delta$ session", fontsize=LABEL_FS, fontweight="bold")
        axC.set_ylabel("correlation (r)", fontsize=LABEL_FS, fontweight="bold")
        axC.set_xlim(1, ns - 1)
        axC.set_ylim(curve_ymin, curve_ymax)
        axC.set_xticks([1, ns - 1])
        axC.set_xticklabels([1, ns - 1], fontsize=TICK_FS, fontweight="bold")
        axC.tick_params(axis="y", labelsize=TICK_FS)
        for t in axC.get_yticklabels():
            t.set_fontweight("bold")

        # -------------------------
        # RIGHT: permutation histogram
        # -------------------------
        counts, bins = np.histogram(
            perm_r[np.isfinite(perm_r)],
            bins=HIST_BINS
        )
        probs = counts / max(counts.sum(), 1)

        axD.bar(
            bins[:-1],
            probs,
            width=np.diff(bins),
            align="edge",
            color="lightgray",
            edgecolor="none"
        )
        axD.axvline(obs_r, color="black", linewidth=LINE_WIDE)

        axD.set_title(
            f"{VREG_NAMES[vreg]} | permutations",
            fontsize=SUBTITLE_FS,
            fontweight="bold"
        )
        axD.set_xlabel(
            f"{CORR_METHOD.capitalize()} correlation (r)",
            fontsize=LABEL_FS,
            fontweight="bold"
        )
        axD.set_ylabel("perm. prob.", fontsize=LABEL_FS, fontweight="bold")
        axD.set_xlim(hist_xmin, hist_xmax)
        axD.tick_params(axis="both", labelsize=TICK_FS)
        for t in axD.get_xticklabels() + axD.get_yticklabels():
            t.set_fontweight("bold")

    NORMALIZATION_LABELS = {
        0: "",
        1: "zscore",
        2: "zeroMean",
        3: "equalStd",
        4: "zeroROImean"
    }

    fig.suptitle(
        f"Population responses drift across sessions",
        y=0.995,
        fontsize=TITLE_FS,
        fontweight="bold"
    )

    if TO_ZSCORE != 0:
        fig.text(
            0.02,
            0.97,
            f"Normalization: {NORMALIZATION_LABELS[TO_ZSCORE]}",
            ha="left",
            va="top",
            fontsize=12,
            fontweight="bold"
        )

    fig.subplots_adjust(
        left=0.06,
        right=0.96,
        bottom=0.04,
        top=0.91,
        wspace=0.05,
        hspace=0.55
    )

    out_path = os.path.join(
        FIG_DIR,
        f"Fig3_population_allROIs{_zstr(TO_ZSCORE)}_{CORR_METHOD}.png"
    )

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {out_path}")
    plt.show()
    plt.close(fig)

# compare drift in population

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================
BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
TO_ZSCORE = 0
NPERMS = 1000
ROI_LIST = [1, 2, 3, 4]
ROI_LABELS = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}
SUBJECTS = list(range(1, 9))
OUT_DIR = os.path.join(BASE_FOLDER, "plots")
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def _zstr(z):
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][z]

def perm_pop_path(base_folder, vreg, isub, to_zscore, nperms):
    return os.path.join(
        base_folder,
        f"permPop_N{nperms}_v{vreg}_sub{isub}{_zstr(to_zscore)}_spearman.npz"
    )

def safe_corr(x, y):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan
    return np.corrcoef(x[m], y[m])[0, 1]

def safe_slope_ols(x, y, add_intercept=True):
    """
    OLS slope for y ~ x (+ intercept), ignoring NaNs/Infs.
    Returns slope (beta1).
    """
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan

    xm = x[m].astype(float)
    ym = y[m].astype(float)

    if add_intercept:
        X = np.column_stack([np.ones_like(xm), xm])
        b, *_ = np.linalg.lstsq(X, ym, rcond=None)
        return float(b[1])
    else:
        denom = np.sum(xm**2)
        if denom == 0:
            return np.nan
        return float(np.sum(xm * ym) / denom)

def bar_with_subject_dots(ax, values_by_roi, roi_names, ylabel, title):
    nroi = len(values_by_roi)
    nsub = len(values_by_roi[0])

    means = [np.nanmean(v) for v in values_by_roi]
    sems  = [np.nanstd(v, ddof=1) / np.sqrt(np.sum(np.isfinite(v))) for v in values_by_roi]

    x = np.arange(nroi)

    ax.bar(
        x, means, yerr=sems, capsize=4,
        facecolor="none", edgecolor="black", linewidth=2,
        zorder=1
    )

    cmap = plt.get_cmap("tab10")
    subject_colors = [cmap(i) for i in range(nsub)]

    data_matrix = np.column_stack(values_by_roi)
    rng = np.random.default_rng(0)

    for roi_idx in range(nroi):
        jitter = rng.uniform(-0.10, 0.10, size=nsub)
        for si in range(nsub):
            ax.scatter(
                x[roi_idx] + jitter[si],
                data_matrix[si, roi_idx],
                s=60,
                color=subject_colors[si],
                edgecolor="black",
                linewidth=0.6,
                alpha=0.95,
                zorder=3
            )

    ax.set_xticks(x)
    ax.set_xticklabels(roi_names, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=16, fontweight="bold")
    ax.set_title(title, fontsize=18, fontweight="bold")
    ax.tick_params(axis="both", labelsize=13)
    ax.axhline(0, linewidth=1.5, color="black", zorder=0)

# ============================================================
# LOAD ONE ROI
# ============================================================
def load_population_perm_for_roi(roi, nperms=1000, to_zscore=0, subjects=SUBJECTS):
    """
    Load population permPop files for one ROI across subjects.

    Returns:
      data_real : (nsub, S, S) avgImgCorrMat
      data_perm : (nsub, nperms, S, S) imgCorrMatPerm
      subjects_found : list
      S : common session count
    """
    mats = []
    perms = []
    subjects_found = []
    min_sessions = None
    min_nperms = None

    for isub in subjects:
        path = perm_pop_path(BASE_FOLDER, roi, isub, to_zscore, nperms)
        if not os.path.exists(path):
            print(f"[WARN] Missing: {path}")
            continue

        d = np.load(path, allow_pickle=True)
        mat = np.array(d["avgImgCorrMat"], dtype=float)
        perm = np.array(d["imgCorrMatPerm"], dtype=float)
        ns = int(d["minSessions"])

        if min_sessions is None:
            min_sessions = ns
        else:
            min_sessions = min(min_sessions, ns)

        if min_nperms is None:
            min_nperms = perm.shape[0]
        else:
            min_nperms = min(min_nperms, perm.shape[0])

        mats.append(mat)
        perms.append(perm)
        subjects_found.append(isub)

    if len(subjects_found) == 0:
        raise RuntimeError(f"No files found for ROI {roi}")

    S = min_sessions
    P = min(min_nperms, nperms)

    mats = [m[:S, :S] for m in mats]
    perms = [p[:P, :S, :S] for p in perms]

    data_real = np.stack(mats, axis=0)   # (nsub, S, S)
    data_perm = np.stack(perms, axis=0)  # (nsub, P, S, S)

    return data_real, data_perm, subjects_found, S

# ============================================================
# SUBJECT-WISE DRIFT METRICS
# ============================================================
def subjectwise_distcorr_and_slope_population(sim_data, sim_perm=None):
    """
    sim_data: (nsub, S, S) real population similarity matrices
    sim_perm: (nsub, nperms, S, S) permuted matrices

    Returns:
      r_sub, slope_sub, p_sub
    """
    nsub, S, _ = sim_data.shape
    sess = np.arange(1, S + 1)
    sessDist = np.abs(sess[:, None] - sess[None, :])
    mask_off = sessDist > 0
    dists_vec = sessDist[mask_off].astype(float)

    r_sub = np.full(nsub, np.nan, float)
    slope_sub = np.full(nsub, np.nan, float)

    p_sub = None
    if sim_perm is not None:
        nperms_local = sim_perm.shape[1]
        p_sub = np.full(nsub, np.nan, float)

    for si in range(nsub):
        M = sim_data[si]
        y_vec = M[mask_off].astype(float)

        r_sub[si] = safe_corr(dists_vec, y_vec)
        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        if sim_perm is not None:
            r_perm_sub = np.full(nperms_local, np.nan, float)
            for ip in range(nperms_local):
                Mp = sim_perm[si, ip]
                r_perm_sub[ip] = safe_corr(dists_vec, Mp[mask_off])

            # one-sided: drift = more negative than null
            k = np.sum(r_perm_sub <= r_sub[si])
            p_sub[si] = (k + 1) / (nperms_local + 1)

    return r_sub, slope_sub, p_sub

def slope_perm_subjectwise_population(sim_data, sim_perm):
    """
    Group-level permutation test for mean slope across subjects.
    """
    nsub, S, _ = sim_data.shape
    nperms = sim_perm.shape[1]

    sess = np.arange(1, S + 1)
    sessDist = np.abs(sess[:, None] - sess[None, :])
    mask_off = sessDist > 0
    dists_vec = sessDist[mask_off].astype(float)

    slope_sub = np.full(nsub, np.nan, float)
    slope_perm_sub = np.full((nsub, nperms), np.nan, float)

    for si in range(nsub):
        M = sim_data[si]
        y_vec = M[mask_off].astype(float)
        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        for ip in range(nperms):
            Mp = sim_perm[si, ip]
            slope_perm_sub[si, ip] = safe_slope_ols(dists_vec, Mp[mask_off], add_intercept=True)

    slope_real = float(np.nanmean(slope_sub))
    slope_perm = np.nanmean(slope_perm_sub, axis=0)

    k = np.sum(slope_perm <= slope_real)
    p_perm = (k + 1) / (nperms + 1)

    return slope_sub, slope_real, slope_perm, p_perm

# ============================================================
# COMPUTE ALL ROIS
# ============================================================
r_metric_by_roi = []
slope_metric_by_roi = []
p_metric_by_roi = []

for roi in ROI_LIST:
    sim_data, sim_perm, subjects_found, S = load_population_perm_for_roi(
        roi=roi,
        nperms=NPERMS,
        to_zscore=TO_ZSCORE,
        subjects=SUBJECTS,
    )

    r_sub, slope_sub, p_sub = subjectwise_distcorr_and_slope_population(
        sim_data, sim_perm
    )

    slope_sub2, slope_real_mean, slope_perm_mean, slope_p_perm = slope_perm_subjectwise_population(
        sim_data, sim_perm
    )

    print(f"{ROI_LABELS[roi]} | mean slope={slope_real_mean:.6g} | slope perm p(one-sided)={slope_p_perm:.4g}")

    r_metric_by_roi.append(r_sub)
    slope_metric_by_roi.append(slope_sub)
    p_metric_by_roi.append(p_sub)

# ============================================================
# PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

bar_with_subject_dots(
    axes[0],
    r_metric_by_roi,
    [ROI_LABELS[r] for r in ROI_LIST],
    ylabel="corr(population similarity, session lag)",
    title="Population drift: correlation with lag"
)

bar_with_subject_dots(
    axes[1],
    slope_metric_by_roi,
    [ROI_LABELS[r] for r in ROI_LIST],
    ylabel="slope of population similarity ~ session lag",
    title="Population drift: slope vs lag"
)

from matplotlib.lines import Line2D

nsub = len(r_metric_by_roi[0])
cmap = plt.get_cmap("tab10")

legend_handles = [
    Line2D(
        [0], [0],
        marker='o',
        color='w',
        markerfacecolor=cmap(i),
        markeredgecolor='black',
        markersize=9,
        label=f"Subject {i+1}"
    )
    for i in range(nsub)
]

fig.legend(
    handles=legend_handles,
    title="Subjects",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.06),
    ncol=4,
    frameon=False,
    fontsize=14,
    title_fontsize=16
)

fig.legends[0].get_title().set_fontweight("bold")
for txt in fig.legends[0].get_texts():
    txt.set_fontweight("normal")

plt.tight_layout(rect=[0, 0, 1, 0.90])

save_path = os.path.join(OUT_DIR, "ROI_population_drift_comparison_subjectcolors_meanline.png")
plt.savefig(save_path, dpi=300, bbox_inches="tight")
print("Saved figure to:", save_path)

plt.show()

# Rdm Plot

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from google.colab import drive
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

drive.mount("/content/drive", force_remount=True)

# =========================================================
# CONFIG
# =========================================================
BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
FIG_DIR = "/content/drive/MyDrive/V1_Drift/plots/rdm_figures"
os.makedirs(FIG_DIR, exist_ok=True)
TO_ZSCORE = 3
CORR_METHOD = "pearson" # pearson or spearman
NPERMS_WANT = 1000

VREGS = [1, 2, 3, 4]
VREG_NAMES = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}
SUBJECTS = list(range(1, 9))

RDM_COMBOS = [
    ("correlation", "spearman"),
    ("correlation", "rmse"),
    ("euclidean", "spearman"),
    ("euclidean", "rmse"),
]

FIG_W = 11
FIG_H = 12
HIST_BINS = 20

LINE_WIDE = 2.5
LINE_NARROW = 1.2


# =========================================================
# HELPERS
# =========================================================
def _zstr(z):
    return ['', '_zscore', '_zeroMean', '_equalStd', '_zeroROImean'][z]


def metric_tag(rdm_build_metric, rdm_compare_metric):
    build_tag = {
        "correlation": "rdmCorrDist",
        "euclidean": "rdmEuclidean"
    }[rdm_build_metric]

    compare_tag = {
        "spearman": "cmpSpearman",
        "rmse": "cmpRMSE"
    }[rdm_compare_metric]

    return f"{build_tag}_{compare_tag}"


def pretty_combo_name(rdm_build_metric, rdm_compare_metric):
    build_name = {
        "correlation": "RDM: 1 - Pearson",
        "euclidean": "RDM: Euclidean"
    }[rdm_build_metric]

    compare_name = {
        "spearman": "Compare: Spearman",
        "rmse": "Compare: RMSE"
    }[rdm_compare_metric]

    return f"{build_name}; {compare_name}"


def perm_pop_path(base_folder, vreg, isub, to_zscore, nperms,
                  rdm_build_metric, rdm_compare_metric):
    tag = metric_tag(rdm_build_metric, rdm_compare_metric)

    return os.path.join(
        base_folder,
        f"permPop_N{nperms}_v{vreg}_sub{isub}"
        f"{_zstr(to_zscore)}_{tag}.npz"
    )


def safe_corr(x, y, method=CORR_METHOD):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    m = np.isfinite(x) & np.isfinite(y)
    if m.sum() < 3:
        return np.nan

    x = x[m]
    y = y[m]

    if method == "pearson":
        return pearsonr(x, y)[0]
    elif method == "spearman":
        return spearmanr(x, y)[0]

    raise ValueError(f"Unknown method: {method}")


def matrix_distance_correlation(mat, dist_matrix):
    """
    Correlate off-diagonal matrix entries with session distance.
    This returns the drift summary r.
    """
    mask = np.isfinite(mat) & (dist_matrix > 0)
    if mask.sum() < 3:
        return np.nan

    return safe_corr(dist_matrix[mask], mat[mask])


def load_region_group(vreg, rdm_build_metric, rdm_compare_metric,
                      subjects=SUBJECTS, base_folder=BASE_FOLDER,
                      to_zscore=TO_ZSCORE, nperms=NPERMS_WANT):
    """
    Load all subjects for one visual region and one RDM build × compare combo.
    """
    mats = []
    lag_curves = []
    perm_mats = []
    found_subjects = []
    min_sessions = None

    for isub in subjects:
        path = perm_pop_path(
            base_folder=base_folder,
            vreg=vreg,
            isub=isub,
            to_zscore=to_zscore,
            nperms=nperms,
            rdm_build_metric=rdm_build_metric,
            rdm_compare_metric=rdm_compare_metric
        )

        if not os.path.exists(path):
            print(f"[WARN] Missing: {path}")
            continue

        d = np.load(path, allow_pickle=True)

        mat = np.array(d["betweenSessCorrOri"], dtype=float)
        lag = np.array(d["betweenSessDistOri"], dtype=float)
        pmat = np.array(d["betweenSessCorrOriPerm"], dtype=float)
        ns = int(d["minSessions"])

        if min_sessions is None:
            min_sessions = ns
        else:
            min_sessions = min(min_sessions, ns)

        mats.append(mat)
        lag_curves.append(lag)
        perm_mats.append(pmat)
        found_subjects.append(isub)

    if len(found_subjects) == 0:
        raise RuntimeError(
            f"No files found for vreg={vreg}, "
            f"build={rdm_build_metric}, compare={rdm_compare_metric}"
        )

    mats = [m[:min_sessions, :min_sessions] for m in mats]
    lag_curves = [l[:min_sessions - 1] for l in lag_curves]
    perm_mats = [p[:, :min_sessions, :min_sessions] for p in perm_mats]

    mats = np.stack(mats, axis=0)
    lag_curves = np.stack(lag_curves, axis=0)
    perm_mats = np.stack(perm_mats, axis=0)

    dist_matrix = np.abs(
        np.subtract.outer(np.arange(min_sessions), np.arange(min_sessions))
    )

    return {
        "subjects": np.array(found_subjects),
        "ns": min_sessions,
        "dist_matrix": dist_matrix,
        "betweenSessCorr_S": mats,
        "betweenSessDist_S": lag_curves,
        "betweenSessCorrPerm_S": perm_mats,
    }


def format_p_value(k, p_perm):
    if k == 0 or p_perm < 0.001:
        return "p < 0.001"
    return f"p = {p_perm:.3f}"


def get_axis_labels_and_p_direction(rdm_compare_metric):

    if rdm_compare_metric == "spearman":
        return {
            "matrix_title": "RDM similarity matrix",
            "ylabel": "RDM similarity (Spearman)",
            "hist_xlabel":
                f"Correlation ({CORR_METHOD})",
            "p_tail": "negative",
        }

    elif rdm_compare_metric == "rmse":
        return {
            "matrix_title": "RDM distance matrix",
            "ylabel": "RDM distance (RMSE)",
            "hist_xlabel":
                f"Correlation ({CORR_METHOD})",
            "p_tail": "positive",
        }


# =========================================================
# PLOT ONE COMBO
# =========================================================
def plot_one_rdm_combo(rdm_build_metric, rdm_compare_metric):
    labels = get_axis_labels_and_p_direction(rdm_compare_metric)

    fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    constrained_layout=False
    )

    gs = fig.add_gridspec(
        nrows=len(VREGS),
        ncols=3,
        width_ratios=[1,1,1],
        wspace=0.05,
        hspace=0.55
    )


    for row, vreg in enumerate(VREGS):
        agg = load_region_group(
            vreg=vreg,
            rdm_build_metric=rdm_build_metric,
            rdm_compare_metric=rdm_compare_metric
        )

        subjects = agg["subjects"]
        ns = agg["ns"]
        dist_matrix = agg["dist_matrix"]
        mats = agg["betweenSessCorr_S"]
        lag_curves = agg["betweenSessDist_S"]
        perm_mats = agg["betweenSessCorrPerm_S"]
        nperms = perm_mats.shape[1]

        # -----------------------------
        # Panel 1: matrix
        # -----------------------------
        ax1 = fig.add_subplot(gs[row,0])

        mean_mat = np.nanmean(mats, axis=0)
        mat_for_show = mean_mat.copy()
        np.fill_diagonal(mat_for_show, np.nan)

        im = ax1.imshow(mat_for_show, origin="upper", aspect="equal", cmap="viridis")
        ax1.set_title(
            f"{VREG_NAMES[vreg]}  {labels['matrix_title']}: mean",
            fontweight="bold"
        )
        ax1.set_xlabel("session", fontweight="bold")
        ax1.set_ylabel("session", fontweight="bold")
        ax1.set_xticks([0, ns - 1])
        ax1.set_xticklabels(["1", str(ns)])
        ax1.set_yticks([0, ns - 1])
        ax1.set_yticklabels(["1", str(ns)])
        ax_cbar = inset_axes(
            ax1,
            width="5%",
            height="100%",
            loc="lower left",
            bbox_to_anchor=(1.05,0,1,1),
            bbox_transform=ax1.transAxes,
            borderpad=0
        )

        cbar = fig.colorbar(im, cax=ax_cbar)

        cbar.ax.tick_params(
            labelsize=8,
            length=2
        )

        for t in cbar.ax.get_yticklabels():
            t.set_fontweight("bold")

        # -----------------------------
        # Panel 2: lag curve
        # -----------------------------
        ax2 = fig.add_subplot(gs[row,1])
        lags = np.arange(1, ns)

        for i in range(lag_curves.shape[0]):
            ax2.plot(lags, lag_curves[i], linewidth=LINE_NARROW, alpha=0.8)

        mean_curve = np.nanmean(lag_curves, axis=0)
        ax2.plot(lags, mean_curve, color="black", linewidth=LINE_WIDE)

        subj_r = np.array([
            matrix_distance_correlation(m, dist_matrix)
            for m in mats
        ])
        obs_r = np.nanmean(subj_r)

        perm_r = np.full((nperms,), np.nan)

        for ip in range(nperms):
            subj_perm_r = np.array([
                matrix_distance_correlation(perm_mats[si, ip], dist_matrix)
                for si in range(len(subjects))
            ])
            perm_r[ip] = np.nanmean(subj_perm_r)

        subj_perm_r_all = np.full((len(subjects), nperms), np.nan)

        for si in range(len(subjects)):
            for ip in range(nperms):
                subj_perm_r_all[si, ip] = matrix_distance_correlation(
                    perm_mats[si, ip],
                    dist_matrix
                )

        if labels["p_tail"] == "negative":
            sub_p = np.array([
                np.mean(subj_perm_r_all[si, :] <= subj_r[si])
                for si in range(len(subjects))
            ])
        else:
            sub_p = np.array([
                np.mean(subj_perm_r_all[si, :] >= subj_r[si])
                for si in range(len(subjects))
            ])

        n_sig_subjects = int(np.sum(sub_p < 0.05))

        # Direction of one-sided permutation test
        if labels["p_tail"] == "negative":
            # drift = similarity decreases with lag
            k = np.sum(perm_r <= obs_r)
        else:
            # drift = distance increases with lag
            k = np.sum(perm_r >= obs_r)

        p_perm = (k + 1) / (nperms + 1)
        p_txt = format_p_value(k, p_perm)

        if p_perm < 0.001:
            sig_txt = "***"
        elif p_perm < 0.01:
            sig_txt = "**"
        elif p_perm < 0.05:
            sig_txt = "*"
        else:
            sig_txt = "n.s."

        ax2.set_title(
            f"{VREG_NAMES[vreg]}  r={obs_r:.2f}  {sig_txt}\n"
            f"{p_txt} | sig subjects: {n_sig_subjects}/{len(subjects)}",
            fontweight="bold"
        )
        ax2.set_xlabel(r"$\Delta$ session", fontweight="bold")
        ax2.set_ylabel(labels["ylabel"], fontweight="bold")
        ax2.set_xlim(1, ns - 1)

        # Better automatic y limits per metric
        y_all = lag_curves[np.isfinite(lag_curves)]
        if y_all.size > 0:
            ymin, ymax = np.nanmin(y_all), np.nanmax(y_all)
            pad = 0.08 * (ymax - ymin) if ymax > ymin else 0.05
            ax2.set_ylim(ymin - pad, ymax + pad)

        # -----------------------------
        # Panel 3: permutation histogram
        # -----------------------------
        ax3 = fig.add_subplot(gs[row,2])
        for ax in [ax1, ax2, ax3]:
            ax.set_box_aspect(1)

        counts, bins = np.histogram(perm_r[np.isfinite(perm_r)], bins=HIST_BINS)
        probs = counts / max(counts.sum(), 1)

        ax3.bar(
            bins[:-1],
            probs,
            width=np.diff(bins),
            align="edge",
            color="lightgray",
            edgecolor="none"
        )
        ax3.axvline(obs_r, color="black", linewidth=LINE_WIDE)

        ax3.set_title(f"{VREG_NAMES[vreg]}  permutations", fontweight="bold")
        ax3.set_xlabel(labels["hist_xlabel"], fontweight="bold")
        ax3.set_ylabel("perm. prob.", fontweight="bold")

    combo_title = pretty_combo_name(rdm_build_metric, rdm_compare_metric)
    tag = metric_tag(rdm_build_metric, rdm_compare_metric)

    NORMALIZATION_LABELS = {
    0: "",
    1: "zscore",
    2: "zeroMean",
    3: "equalStd",
    4: "zeroROImean"
    }

    plt.suptitle(
        f"RDM drift across sessions: {combo_title}",
        y=0.995,
        fontsize=18,
        fontweight="bold"
    )

    if TO_ZSCORE != 0:
      fig.text(
          0.02, 0.965,
          f"Normalization: {NORMALIZATION_LABELS[TO_ZSCORE]}",
          ha="left",
          va="top",
          fontsize=12,
          fontweight="bold"
          )

    fig.subplots_adjust(
    left=0.06,
    right=0.96,
    bottom=0.04,
    top=0.91,
    wspace=0.05,
    hspace=0.55
    )

    out_path = os.path.join(
        FIG_DIR,
        f"Fig3_RDM_allROIs{_zstr(TO_ZSCORE)}_{tag}_{CORR_METHOD}.png"
        )

    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    print(f"Figure saved to: {out_path}")

    plt.show()
    plt.close(fig)


# =========================================================
# RUN ALL COMBOS
# =========================================================
for rdm_build_metric, rdm_compare_metric in RDM_COMBOS:
    print("\n" + "=" * 80)
    print(f"Plotting combo: build={rdm_build_metric}, compare={rdm_compare_metric}")
    print("=" * 80)

    try:
        plot_one_rdm_combo(rdm_build_metric, rdm_compare_metric)
        plt.close("all")
    except Exception as e:
        print(f"[ERROR] Failed combo build={rdm_build_metric}, compare={rdm_compare_metric}: {e}")

# compare drift in rdm

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ============================================================
# CONFIG
# ============================================================
BASE_FOLDER = "/content/drive/MyDrive/V1_Drift/repDrift_expand"
TO_ZSCORE = 0
NPERMS = 1000
RDM_METRIC = "rmse"   # "spearman" or "rmse"
ROI_LIST = [1, 2, 3, 4]
ROI_LABELS = {1: "V1", 2: "V2", 3: "V3", 4: "hV4"}
SUBJECTS = list(range(1, 9))

OUT_DIR = os.path.join(BASE_FOLDER, "plots")
os.makedirs(OUT_DIR, exist_ok=True)

FIG_W = 14
FIG_H = 6

# ============================================================
# HELPERS
# ============================================================
def _zstr(z):
    return ["", "_zscore", "_zeroMean", "_equalStd", "_zeroROImean"][z]


def perm_pop_path(base_folder, vreg, isub, to_zscore, nperms, metric=RDM_METRIC):
    return os.path.join(
        base_folder,
        f"permPop_N{nperms}_v{vreg}_sub{isub}{_zstr(to_zscore)}_{metric}.npz"
    )


def safe_corr(x, y):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3:
        return np.nan

    return np.corrcoef(x[m], y[m])[0, 1]


def safe_slope_ols(x, y, add_intercept=True):
    """
    OLS slope for y ~ x, ignoring NaNs/Infs.
    Returns slope beta.
    """
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()

    m = np.isfinite(x) & np.isfinite(y)

    if m.sum() < 3:
        return np.nan

    xm = x[m].astype(float)
    ym = y[m].astype(float)

    if add_intercept:
        X = np.column_stack([np.ones_like(xm), xm])
        b, *_ = np.linalg.lstsq(X, ym, rcond=None)
        return float(b[1])

    denom = np.sum(xm ** 2)
    if denom == 0:
        return np.nan

    return float(np.sum(xm * ym) / denom)


def get_metric_name(path):
    path_lower = path.lower()

    if "spearman" in path_lower:
        return "Spearman"
    elif "pearson" in path_lower:
        return "Pearson"
    elif "rmse" in path_lower:
        return "RMSE"
    else:
        return "Unknown metric"


def format_p_value(p):
    if not np.isfinite(p):
        return "p = NaN"
    elif p < 0.001:
        return "p < 0.001"
    else:
        return f"p = {p:.3f}"


# ============================================================
# LOAD RDM DATA
# ============================================================
def load_rdm_perm_for_roi(roi, nperms=1000, to_zscore=0, subjects=SUBJECTS):
    """
    Load RDM drift data from permPop files.

    Uses:
      real matrix: betweenSessCorr
      perm matrix: betweenSessCorrPerm

    Returns:
      data_real: (nsub, S, S)
      data_perm: (nsub, P, S, S)
      subjects_found: list
      S: common session count
    """
    mats = []
    perms = []
    subjects_found = []

    min_sessions = None
    min_nperms = None

    for isub in subjects:
        path = perm_pop_path(BASE_FOLDER, roi, isub, to_zscore, nperms, metric=RDM_METRIC)
        if not os.path.exists(path):
            print(f"[WARN] Missing: {path}")
            continue

        d = np.load(path, allow_pickle=True)

        mat = np.array(d["betweenSessCorrOri"], dtype=float)
        perm = np.array(d["betweenSessCorrOriPerm"], dtype=float)
        ns = int(d["minSessions"])

        min_sessions = ns if min_sessions is None else min(min_sessions, ns)
        min_nperms = perm.shape[0] if min_nperms is None else min(min_nperms, perm.shape[0])

        mats.append(mat)
        perms.append(perm)
        subjects_found.append(isub)

    if len(subjects_found) == 0:
        raise RuntimeError(f"No files found for ROI {roi}")

    S = min_sessions
    P = min(min_nperms, nperms)

    mats = [m[:S, :S] for m in mats]
    perms = [p[:P, :S, :S] for p in perms]

    data_real = np.stack(mats, axis=0)
    data_perm = np.stack(perms, axis=0)

    return data_real, data_perm, subjects_found, S


# ============================================================
# SUBJECT-WISE RDM DRIFT METRICS
# ============================================================
def subjectwise_distcorr_and_slope_rdm(rdm_data, rdm_perm=None):
    """
    Computes subject-wise:
      1. correlation between RDM similarity and session lag
      2. slope of RDM similarity ~ session lag
      3. subject-wise permutation p-values for correlation
    """
    nsub, S, _ = rdm_data.shape

    sess = np.arange(1, S + 1)
    sess_dist = np.abs(sess[:, None] - sess[None, :])

    mask_off = sess_dist > 0
    dists_vec = sess_dist[mask_off].astype(float)

    r_sub = np.full(nsub, np.nan)
    slope_sub = np.full(nsub, np.nan)

    p_sub = None

    if rdm_perm is not None:
        nperms_local = rdm_perm.shape[1]
        p_sub = np.full(nsub, np.nan)

    for si in range(nsub):
        M = rdm_data[si]
        y_vec = M[mask_off].astype(float)

        r_sub[si] = safe_corr(dists_vec, y_vec)
        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        if rdm_perm is not None:
            r_perm_sub = np.full(nperms_local, np.nan)

            for ip in range(nperms_local):
                Mp = rdm_perm[si, ip]
                r_perm_sub[ip] = safe_corr(dists_vec, Mp[mask_off])

            # one-sided test: drift = more negative than null
            k = np.sum(r_perm_sub <= r_sub[si])
            p_sub[si] = (k + 1) / (nperms_local + 1)

    return r_sub, slope_sub, p_sub


def slope_perm_subjectwise_rdm(rdm_data, rdm_perm):
    """
    Group-level permutation test for mean slope across subjects.
    """
    nsub, S, _ = rdm_data.shape
    nperms = rdm_perm.shape[1]

    sess = np.arange(1, S + 1)
    sess_dist = np.abs(sess[:, None] - sess[None, :])

    mask_off = sess_dist > 0
    dists_vec = sess_dist[mask_off].astype(float)

    slope_sub = np.full(nsub, np.nan)
    slope_perm_sub = np.full((nsub, nperms), np.nan)

    for si in range(nsub):
        M = rdm_data[si]
        y_vec = M[mask_off].astype(float)

        slope_sub[si] = safe_slope_ols(dists_vec, y_vec, add_intercept=True)

        for ip in range(nperms):
            Mp = rdm_perm[si, ip]
            slope_perm_sub[si, ip] = safe_slope_ols(
                dists_vec,
                Mp[mask_off],
                add_intercept=True
            )

    slope_real = float(np.nanmean(slope_sub))
    slope_perm = np.nanmean(slope_perm_sub, axis=0)

    # one-sided test: drift = more negative slope than null
    k = np.sum(slope_perm <= slope_real)
    p_perm = (k + 1) / (nperms + 1)

    return slope_sub, slope_real, slope_perm, p_perm


# ============================================================
# PLOT HELPER
# ============================================================
def bar_with_subject_dots(ax, values_by_roi, roi_names, ylabel, title):
    nroi = len(values_by_roi)
    nsub = len(values_by_roi[0])

    means = [
        np.nanmean(v)
        for v in values_by_roi
    ]

    sems = [
        np.nanstd(v, ddof=1) / np.sqrt(np.sum(np.isfinite(v)))
        for v in values_by_roi
    ]

    x = np.arange(nroi)

    ax.bar(
        x,
        means,
        yerr=sems,
        capsize=4,
        facecolor="none",
        edgecolor="black",
        linewidth=2,
        zorder=1
    )

    cmap = plt.get_cmap("tab10")
    subject_colors = [cmap(i) for i in range(nsub)]

    data_matrix = np.column_stack(values_by_roi)
    rng = np.random.default_rng(0)

    for roi_idx in range(nroi):
        jitter = rng.uniform(-0.10, 0.10, size=nsub)

        for si in range(nsub):
            ax.scatter(
                x[roi_idx] + jitter[si],
                data_matrix[si, roi_idx],
                s=60,
                color=subject_colors[si],
                edgecolor="black",
                linewidth=0.6,
                alpha=0.95,
                zorder=3
            )

    ax.set_xticks(x)
    ax.set_xticklabels(roi_names, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=15, fontweight="bold")
    ax.set_title(title, fontsize=17, fontweight="bold")
    ax.tick_params(axis="both", labelsize=13)
    ax.axhline(0, linewidth=1.5, color="black", zorder=0)


# ============================================================
# COMPUTE ALL ROIS
# ============================================================
r_metric_by_roi = []
slope_metric_by_roi = []
p_metric_by_roi = []
slope_p_by_roi = []

example_path = perm_pop_path(BASE_FOLDER, ROI_LIST[0], SUBJECTS[0], TO_ZSCORE, NPERMS, metric=RDM_METRIC)
metric_name = get_metric_name(example_path)

for roi in ROI_LIST:
    rdm_data, rdm_perm, subjects_found, S = load_rdm_perm_for_roi(
        roi=roi,
        nperms=NPERMS,
        to_zscore=TO_ZSCORE,
        subjects=SUBJECTS,
    )

    r_sub, slope_sub, p_sub = subjectwise_distcorr_and_slope_rdm(
        rdm_data,
        rdm_perm
    )

    slope_sub2, slope_real_mean, slope_perm_mean, slope_p_perm = slope_perm_subjectwise_rdm(
        rdm_data,
        rdm_perm
    )

    print(
        f"{ROI_LABELS[roi]} | "
        f"mean r={np.nanmean(r_sub):.3f} | "
        f"mean RDM slope={slope_real_mean:.6g} | "
        f"slope perm p(one-sided)={format_p_value(slope_p_perm)}"
    )

    r_metric_by_roi.append(r_sub)
    slope_metric_by_roi.append(slope_sub)
    p_metric_by_roi.append(p_sub)
    slope_p_by_roi.append(slope_p_perm)


# ============================================================
# PLOT
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, FIG_H))

bar_with_subject_dots(
    axes[0],
    r_metric_by_roi,
    [ROI_LABELS[r] for r in ROI_LIST],
    ylabel="corr(RDM similarity, session lag)",
    title=f"RDM drift: correlation with lag ({metric_name})"
)

bar_with_subject_dots(
    axes[1],
    slope_metric_by_roi,
    [ROI_LABELS[r] for r in ROI_LIST],
    ylabel="slope of RDM similarity ~ session lag",
    title=f"RDM drift: slope vs lag ({metric_name})"
)

nsub = len(r_metric_by_roi[0])
cmap = plt.get_cmap("tab10")

legend_handles = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=cmap(i),
        markeredgecolor="black",
        markersize=9,
        label=f"Subject {i + 1}"
    )
    for i in range(nsub)
]

fig.legend(
    handles=legend_handles,
    title="Subjects",
    loc="upper center",
    bbox_to_anchor=(0.5, 1.06),
    ncol=4,
    frameon=False,
    fontsize=14,
    title_fontsize=16
)

fig.legends[0].get_title().set_fontweight("bold")

plt.suptitle(
    f"ROI comparison of RDM drift across sessions ({metric_name})",
    fontsize=18,
    fontweight="bold",
    y=1.13
)

plt.tight_layout(rect=[0, 0, 1, 0.92])

save_path = os.path.join(
    OUT_DIR,
    f"ROI_RDM_drift_comparison_subjectcolors_meanline_{metric_name}.png"
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
print("Saved figure to:", save_path)

plt.show()